# 02 · Seria i uderzenia
**Cel:** utrzymać porządek w danych, sprawdzić każdy podział i wyeksportować cechy. Jeden `plate_id` oznacza jeden fizyczny detal — także gdy nagrasz go w kilku plikach.

In [1]:
from pathlib import Path
import os, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Uruchom notebook z katalogu repo lub notebooks/."
os.chdir(ROOT)
from pinglab.audio import Config, load_audio, detect_hits, cut_hits, listen
from pinglab.demo import generate_demo
if not (ROOT / "data/demo/SYN_001_ok.wav").exists():
    generate_demo(ROOT / "data/demo")
CONFIG = Config()
FMAX = 20000  # wspólne pasmo wszystkich plików; dla źródeł 16 kHz ustaw np. 7500
print("Repo:", ROOT)


Repo: /Users/karolsii/dev/ceramic-ping-lab


In [2]:
# KONFIGURACJA
MANIFEST = ROOT / "data/demo/manifest.csv"
ANNOTATIONS = ROOT / "data/demo/reviewed_annotations.json"
OUT = ROOT / "reports/runs/notebook_batch"


## Manifest
Wiersz = plik. Obowiązkowe: `path`, `plate_id`, `label`, `session_id`. Etykiety: `ok`, `nok`, `unknown`, `background`. Ścieżka względna jest liczona **od pliku CSV**. Szablon: `data/manifest.template.csv`.

`channel` = opcjonalny numer kanału (0/1, puste = auto). `position` = miejsce uderzenia; `environment` = warunki; `notes` = opis wady/podparcia; `photo` = ścieżka zdjęcia. Przy wielu miejscach w jednym pliku wpisz kolejność i zachowuj ten sam protokół.

In [3]:
from pinglab.dataset import read_manifest, analyze_manifest
manifest = read_manifest(MANIFEST)
display(manifest.drop(columns=['resolved_path','sha256']).head(12))
print("Fizyczne płytki:",manifest[manifest.label!='background'].groupby('label').plate_id.nunique().to_dict())

,path,plate_id,label,session_id,position,environment,notes,photo,channel
0,SYN_001_ok.wav,SYN_001,ok,S1,A/B/C,synthetic,"SYMULACJA, nie ceramika klienta",,
1,SYN_002_ok.wav,SYN_002,ok,S2,A/B/C,synthetic,"SYMULACJA, nie ceramika klienta",,
2,SYN_003_ok.wav,SYN_003,ok,S3,A/B/C,synthetic,"SYMULACJA, nie ceramika klienta",,
3,SYN_004_ok.wav,SYN_004,ok,S1,A/B/C,synthetic,"SYMULACJA, nie ceramika klienta",,
4,SYN_005_ok.wav,SYN_005,ok,S2,A/B/C,synthetic,"SYMULACJA, nie ceramika klienta",,
5,SYN_006_ok.wav,SYN_006,ok,S3,A/B/C,synthetic,"SYMULACJA, nie ceramika klienta",,
6,SYN_007_ok.wav,SYN_007,ok,S1,A/B/C,synthetic,"SYMULACJA, nie ceramika klienta",,
7,SYN_008_ok.wav,SYN_008,ok,S2,A/B/C,synthetic,"SYMULACJA, nie ceramika klienta",,
8,SYN_009_ok.wav,SYN_009,ok,S3,A/B/C,synthetic,"SYMULACJA, nie ceramika klienta",,
9,SYN_010_ok.wav,SYN_010,ok,S1,A/B/C,synthetic,"SYMULACJA, nie ceramika klienta",,


Fizyczne płytki: {'nok': 10, 'ok': 20}


## Zatwierdź uderzenia
Wybierz plik → **Wczytaj / wykryj** → odsłuchaj → popraw sekundy → **Podgląd czasów** → **Zatwierdź i zapisz**. Większy próg wykrywa mniej kandydatów. Zatwierdzone czasy mają pierwszeństwo przed automatem; aby przeliczyć je automatycznie, usuń wpis z JSON. Puste pole i zapis oznaczają świadome odrzucenie wszystkich uderzeń.

In [4]:
from pinglab.widgets import review_manifest
review = review_manifest(MANIFEST,ANNOTATIONS,CONFIG)

## Przelicz serię
Komórka działa także przed ręcznym przeglądem, ale wynik zaznaczy niezatwierdzone pliki. Przy zmianie okna / pasma przelicz wszystkie pliki. W notebooku ten katalog wyników jest aktualizowany; do zachowania wersji zmień `OUT`.

In [5]:
events, quality = analyze_manifest(MANIFEST,OUT,CONFIG,FMAX,
                                   ANNOTATIONS if ANNOTATIONS.exists() else None)
display(quality[['path','label','peak_dbfs','clipping_fraction','n_hits','reviewed','warnings']])
print("Uderzenia:", len(events), "· Wyniki:", OUT)
assert not events.empty, "Nie wykryto uderzeń; popraw segmentację."
display(events[['plate_id','label','hit_id','start_s','end_s','truncated']].head())

,path,label,peak_dbfs,clipping_fraction,n_hits,reviewed,warnings
0,SYN_001_ok.wav,ok,-3.956122,0.0,3,True,
1,SYN_002_ok.wav,ok,-3.134742,0.0,3,False,
2,SYN_003_ok.wav,ok,-3.305335,0.0,3,False,
3,SYN_004_ok.wav,ok,-3.730169,0.0,3,False,
4,SYN_005_ok.wav,ok,-4.800968,0.0,3,False,
5,SYN_006_ok.wav,ok,-6.851207,0.0,3,False,
6,SYN_007_ok.wav,ok,-3.547069,0.0,3,False,
7,SYN_008_ok.wav,ok,-5.650114,0.0,3,False,
8,SYN_009_ok.wav,ok,-6.536152,0.0,3,False,
9,SYN_010_ok.wav,ok,-5.302261,0.0,3,False,


Uderzenia: 90 · Wyniki: /Users/karolsii/dev/ceramic-ping-lab/reports/runs/notebook_batch


,plate_id,label,hit_id,start_s,end_s,truncated
0,SYN_001,ok,SYN_001:534ea0236b:1,0.37,1.395,False
1,SYN_001,ok,SYN_001:534ea0236b:2,1.87,2.895,False
2,SYN_001,ok,SYN_001:534ea0236b:3,3.37,4.395,False
3,SYN_002,ok,SYN_002:d53f7c06be:1,0.37,1.395,False
4,SYN_002,ok,SYN_002:d53f7c06be:2,1.87,2.895,False


In [6]:
from pinglab.report import write_report
source_kind = 'synthetic' if events.environment.eq('synthetic').any() else 'client'
report = write_report(events,quality,OUT/'report.html',source_kind=source_kind)
display(Markdown(f"Raport HTML: `{report}`"))

Raport HTML: `/Users/karolsii/dev/ceramic-ping-lab/reports/runs/notebook_batch/report.html`

**Pliki wynikowe:** `quality.csv` — jakość; `features.csv` — cechy każdego uderzenia; `run.json` — parametry i sumy kontrolne. Dalej: **03** — podobieństwa i mały model.